In [0]:
try:
    from pyspark.sql.types import (
        StructType, StructField,
        StringType, DoubleType,
        IntegerType, LongType
    )
 
    schema = StructType([
        StructField('_c0',                    IntegerType(), True),  # índice
        StructField('trans_date_trans_time',  StringType(),  True),  # fecha y hora transacción
        StructField('cc_num',                 LongType(),    True),  # número de tarjeta
        StructField('merchant',               StringType(),  True),  # nombre del comercio
        StructField('category',               StringType(),  True),  # categoría del comercio
        StructField('amt',                    DoubleType(),  True),  # monto de la transacción
        StructField('first',                  StringType(),  True),  # nombre del titular
        StructField('last',                   StringType(),  True),  # apellido del titular
        StructField('gender',                 StringType(),  True),  # género del titular
        StructField('street',                 StringType(),  True),  # calle del titular
        StructField('city',                   StringType(),  True),  # ciudad del titular
        StructField('state',                  StringType(),  True),  # estado del titular
        StructField('zip',                    IntegerType(), True),  # código postal
        StructField('lat',                    DoubleType(),  True),  # latitud del titular
        StructField('long',                   DoubleType(),  True),  # longitud del titular
        StructField('city_pop',               IntegerType(), True),  # población de la ciudad
        StructField('job',                    StringType(),  True),  # ocupación del titular
        StructField('dob',                    StringType(),  True),  # fecha de nacimiento
        StructField('trans_num',              StringType(),  True),  # código único de transacción
        StructField('unix_time',              LongType(),    True),  # timestamp unix
        StructField('merch_lat',              DoubleType(),  True),  # latitud del comercio
        StructField('merch_long',             DoubleType(),  True),  # longitud del comercio
        StructField('is_fraud',               IntegerType(), True),  # 0=legítima, 1=fraude
    ])
 
    print("Esquema definido correctamente — 23 columnas")
    df_temp = spark.createDataFrame([], schema)
    df_temp.printSchema()
 
except Exception as e:
    print(f"Error al definir el esquema: {type(e).__name__}: {str(e)}")
    import traceback
    traceback.print_exc()

In [0]:
try:
    RUTA_CSV = '/Volumes/fraud_project/bronze/raw_data/input/csv/fraudTest.csv'
    DIR_CSV  = '/Volumes/fraud_project/bronze/raw_data/input/csv/'
 
    # Verificar que el archivo existe antes de leerlo
    print("Archivos disponibles en el directorio:")
    archivos = dbutils.fs.ls(DIR_CSV)
    for f in archivos:
        print(f"   • {f.name}  ({f.size:,} bytes)")
 
    # Leer el CSV con el esquema definido
    df = spark.read.format('csv') \
        .option('header', 'true') \
        .schema(schema) \
        .load(RUTA_CSV)
 
    total_registros = df.count()
    print(f"\nArchivo leído correctamente")
    print(f"   Total de registros : {total_registros:,}")
    print(f"   Total de columnas  : {len(df.columns)}")
 
except Exception as e:
    print(f"Error al leer el archivo: {type(e).__name__}: {str(e)}")
    import traceback
    traceback.print_exc()
 

In [0]:
try:
    from pyspark.sql.functions import col
 
    # Guardia: verificar que df esté definido (viene de la celda 2)
    if 'df' not in dir():
        raise Exception("'df' no está definido. Ejecuta primero la Celda 2 (lectura del CSV).")
 
    # Registros VÁLIDOS: is_fraud debe ser exactamente 0 o 1
    df_validos = df.filter(col('is_fraud').isin([0, 1]))
 
    # Registros INVÁLIDOS: campos críticos nulos
    df_invalidos = df.filter(
        col('amt').isNull()                    |
        col('is_fraud').isNull()               |
        col('trans_date_trans_time').isNull()  |
        col('merchant').isNull()
    )
 
    n_validos   = df_validos.count()
    n_invalidos = df_invalidos.count()
 
    print(f"Separación completada")
    print(f"   Registros válidos     : {n_validos:,}")
    print(f"   Registros descartados : {n_invalidos:,}")
 
    if n_invalidos > 0:
        print("\n Muestra de registros inválidos:")
        display(df_invalidos.limit(10))
 
except Exception as e:
    print(f"Error al separar los datos: {type(e).__name__}: {str(e)}")
    import traceback
    traceback.print_exc()

In [0]:
try:
    # Guardia: verificar que los DataFrames estén definidos
    for nombre_var in ['df_validos', 'df_invalidos']:
        if nombre_var not in dir():
            raise Exception(
                f"'{nombre_var}' no está definido. "
                "Ejecuta primero la Celda 3 (separación de registros)."
            )
 
    TABLA_VALIDOS   = 'fraud_project.bronze.transacciones_validas'
    TABLA_INVALIDOS = 'fraud_project.bronze.transacciones_con_errores'
 
    # Guardar registros válidos
    df_validos.write \
        .format('delta') \
        .mode('overwrite') \
        .saveAsTable(TABLA_VALIDOS)
    print(f"Tabla guardada: {TABLA_VALIDOS}")
 
    # Guardar registros inválidos (puede estar vacía, igual se crea)
    df_invalidos.write \
        .format('delta') \
        .mode('overwrite') \
        .saveAsTable(TABLA_INVALIDOS)
    print(f"Tabla guardada: {TABLA_INVALIDOS}")
 
    print(f"\n Registros válidos guardados  : {df_validos.count():,}")
    print(f"   Registros con errores        : {df_invalidos.count():,}")
 
except Exception as e:
    print(f"Error al guardar las tablas: {type(e).__name__}: {str(e)}")
    import traceback
    traceback.print_exc()

In [0]:
try:
    if 'df_validos' not in dir():
        raise Exception("'df_validos' no está definido. Ejecuta primero las celdas anteriores.")
 
    print("=== Vista de los datos válidos (primeras filas) ===")
    display(df_validos)
 
    print("\n=== Distribución por tipo (is_fraud) ===")
    df_validos.groupBy('is_fraud').count() \
        .withColumnRenamed('count', 'total_registros') \
        .show()
 
    print("=== Columnas del dataset ===")
    print(df_validos.columns)
 
except Exception as e:
    print(f"Error al visualizar: {type(e).__name__}: {str(e)}")
    import traceback
    traceback.print_exc()

In [0]:
try:
    for nombre_var in ['df', 'df_validos', 'df_invalidos']:
        if nombre_var not in dir():
            raise Exception(
                f"'{nombre_var}' no definido. Asegúrate de ejecutar todas las celdas anteriores."
            )
 
    total    = df.count()
    validos  = df_validos.count()
    invalidos = df_invalidos.count()
 
    print("=" * 45)
    print("       RESUMEN EJECUTIVO DEL PROCESO        ")
    print("=" * 45)
    print(f"  Total registros leídos    : {total:>10,}")
    print(f"  Registros válidos         : {validos:>10,}")
    print(f"  Registros con errores     : {invalidos:>10,}")
    print(f"  Tasa de calidad           : {validos / total * 100:>9.2f}%")
    print("=" * 45)
    print(f"  Tabla válidos  → fraud_project.bronze.transacciones_validas")
    print(f"  Tabla errores  → fraud_project.bronze.transacciones_con_errores")
    print("=" * 45)
    print(" Pipeline Bronze completado exitosamente")
 
except Exception as e:
    print(f"Error en el resumen: {type(e).__name__}: {str(e)}")
    import traceback
    traceback.print_exc()